# NBO Property Extraction from ORCA Output

This notebook extracts Natural Bond Orbital (NBO) properties from ORCA quantum chemistry output files.

## Approach

The notebook uses a **section-based parsing strategy**:
1. First, locate the entire NBO section in the ORCA output
2. For each property type, find the specific subsection within the NBO output
3. Parse the data within that subsection using regex patterns

## Extracted Data Structures

- `atom_properties`: List indexed by atom number containing charge, core, valence, rydberg populations
- `total_properties`: Dictionary with total charge and electron populations
- `nao_properties`: List of Natural Atomic Orbital occupancies and energies
- `nbo_properties`: List of Natural Bond Orbital summary (type, occupancy, energy)
- `bond_orbital_details`: List of detailed bond orbitals with hybridization and coefficients
- `nbo_directionality`: List of bond bending/directionality information (if present)
- `second_order_perturbation`: List of donor-acceptor interactions (if present)
- `nbo_analysis_info`: Dictionary with NBO analysis convergence information

In [89]:
from pathlib import Path

output_file_to_analyze = Path("./orca_example/test.out")

In [90]:
import re

# Read the ORCA output file
with open(output_file_to_analyze, "r") as f:
    content = f.read()
    
# Find the NBO section
nbo_start = content.find("Now starting NBO")
if nbo_start != -1:
    # Find the end of NBO section (marked by "returned from NBO program")
    nbo_end = content.find("*** returned from  NBO  program ***", nbo_start)
    if nbo_end != -1:
        nbo_section = content[nbo_start:nbo_end + len("*** returned from  NBO  program ***")]
        print(f"Found NBO section: {len(nbo_section)} characters")
    else:
        print("Could not find end of NBO section")
        nbo_section = None
else:
    print("Could not find 'Now starting NBO' in the file")
    nbo_section = None

Found NBO section: 8389 characters


In [98]:
npa_section_lines

['    H  1    0.00000      0.00000     0.99914    0.00086     1.00000',
 '    H  2    0.00000      0.00000     0.99914    0.00086     1.00000',
 ' ====================================================================',
 ' * Total *  0.00000      0.00000     1.99829    0.00171     2.00000',
 '',
 '                                 Natural Population',
 ' ---------------------------------------------------------',
 '   Valence                    1.99829 ( 99.9143% of    2)',
 '   Natural Minimal Basis      1.99829 ( 99.9143% of    2)',
 '   Natural Rydberg Basis      0.00171 (  0.0857% of    2)',
 ' ---------------------------------------------------------',
 '',
 '    Atom No         ']

In [136]:
# Extract Natural Population Analysis
# Find the NPA subsection first
npa_start = nbo_section.find("Summary of Natural Population Analysis:")
npa_end = nbo_section.find("Natural Electron Configuration", npa_start)
if npa_end == -1:
    npa_end = nbo_section.find("\n\n NATURAL BOND ORBITAL ANALYSIS:", npa_start)

if npa_start != -1 and npa_end != -1:
    npa_section = nbo_section[npa_start:npa_end]
    npa_section_lines = npa_section.splitlines()[6:] # Skip header lines
    
    # Initialize arrays
    atom_properties = [None] # 1-indexed, so start with None at index 0
    total_properties = {}

    while "=========================" not in npa_section_lines[len(atom_properties)-1]:
        line = npa_section_lines[len(atom_properties)-1].strip()
        atom_data = line.split()

        atom_properties.append({
            'atom': atom_data[0],
            'charge': float(atom_data[2]),
            'core': float(atom_data[3]),
            'valence': float(atom_data[4]),
            'rydberg': float(atom_data[5]),
            'total': float(atom_data[6])
        })

    total_data = npa_section_lines[len(atom_properties)].strip().split()
    # Extract total properties (line starting with "* Total *")
    total_properties = {
        'charge': float(total_data[3]),
        'core': float(total_data[4]),
        'valence': float(total_data[5]),
        'rydberg': float(total_data[6]),
        'total': float(total_data[7])
    }
    
    print(f"Extracted {len([p for p in atom_properties if p is not None])} atom properties")
    print(f"Total Properties: {total_properties}")
    print(f"\nAtom Properties (indexed by atom number):")
    for i, prop in enumerate(atom_properties):
        if prop is not None:
            print(f"  Atom {i}: {prop}")
else:
    print("Could not find Natural Population Analysis section")
    atom_properties = []
    total_properties = {}

Extracted 2 atom properties
Total Properties: {'charge': 0.0, 'core': 0.0, 'valence': 1.99829, 'rydberg': 0.00171, 'total': 2.0}

Atom Properties (indexed by atom number):
  Atom 1: {'atom': 'H', 'charge': 0.0, 'core': 0.0, 'valence': 0.99914, 'rydberg': 0.00086, 'total': 1.0}
  Atom 2: {'atom': 'H', 'charge': 0.0, 'core': 0.0, 'valence': 0.99914, 'rydberg': 0.00086, 'total': 1.0}


In [170]:
nbo_summary_section.splitlines()

['NATURAL BOND ORBITALS (Summary):',
 '',
 '                                                     Principal Delocalizations',
 '           NBO                 Occupancy    Energy   (geminal,vicinal,remote)',
 ' ===============================================================================',
 ' Molecular unit  1  (H2)',
 ' ------ Lewis --------------------------------------',
 '    1. BD ( 1) H  1- H  2       2.00000    -0.43520',
 ' ------ non-Lewis ----------------------------------',
 '    2. BD*( 1) H  1- H  2       0.00000     0.51844',
 '    3. RY ( 1) H  1             0.00000     1.85236',
 '    4. RY ( 2) H  1             0.00000     0.79278',
 '    5. RY ( 3) H  1             0.00000     1.67050',
 '    6. RY ( 4) H  1             0.00000     1.67091',
 '    7. RY ( 5) H  1             0.00000     2.61649',
 '    8. RY ( 1) H  2             0.00000     2.09016',
 '    9. RY ( 2) H  2             0.00000     1.67050',
 '   10. RY ( 3) H  2             0.00000     1.67091',
 '   

In [201]:
nbo_summary_section.splitlines()[7][5]

'.'

In [203]:
# Extract Natural Atomic Orbital (NAO) occupancies
# Find the NAO subsection
nao_start = nbo_section.find("NATURAL POPULATIONS:  Natural atomic orbital occupancies")
nao_end = nbo_section.find("Summary of Natural Population Analysis:", nao_start)

nao_properties = []

if nao_start != -1 and nao_end != -1:
    nao_subsection = nbo_section[nao_start:nao_end]
    nao_lines = nao_subsection.splitlines()[4:]

    for line in nao_lines:

        if len(line.strip()) == 0:
            continue

        nao_properties.append({
            'orbital_num': int(line[0:5].strip()),
            'atom': line[8:9],
            'atom_num': int(line[9:14].strip()),
            'lang': line[14:21].strip(),
            'type_ao': line[21:29],
            'occupancy': float(line[33:42]),
            'energy': float(line[47:56])
        })

    print(f"Extracted {len(nao_properties)} NAO properties")
else:
    print("Could not find NAO section")
    nao_properties = []


# Extract Natural Bond Orbital (NBO) summary
# Find the NBO summary subsection
nbo_summary_start = nbo_section.find("NATURAL BOND ORBITALS (Summary):")
nbo_summary_end = nbo_section.find("$CHOOSE", nbo_summary_start)
if nbo_summary_end == -1:
    nbo_summary_end = nbo_section.find("Checkpoint orbital", nbo_summary_start)

nbo_properties = []

if nbo_summary_start != -1 and nbo_summary_end != -1:
    nbo_summary_section = nbo_section[nbo_summary_start:nbo_summary_end]
    nbo_summary_lines = nbo_summary_section.splitlines()[7:]

    for line in nbo_summary_lines:
        if len(line.strip()) < 5:
            continue
        if not line[4].isdigit():
            continue

        atom1 = int(line[16:19])
        atom2 = line[22:25].strip()
        if len(atom2) > 0:
            atom2 = int(atom2)
            atom_numbers = [atom1, atom2]
        else:
            atom_numbers = [atom1]

        nbo_properties.append({
            'nbo_num': int(line[0:5]),
            'nbo_type': line[7:10].strip(),
            'bond_index': int(line[12:13]),
            'atoms': atom_numbers,
            'occupancy': float(line[32:39]),
            'energy': float(line[43:56])
        })
    
    print(f"\nExtracted {len(nbo_properties)} NBO properties")
else:
    print("Could not find NBO summary section")
    nbo_properties = []

Extracted 12 NAO properties

Extracted 12 NBO properties


In [184]:
# Extract detailed bond orbital properties with hybridization
# Find the detailed bond orbital subsection
bond_detail_start = nbo_section.find("(Occupancy)   Bond orbital / Coefficients / Hybrids")
lewis_start = nbo_section.find("---------------- Lewis", bond_detail_start)
non_lewis_start = nbo_section.find("---------------- non-Lewis", lewis_start)

bond_orbital_details = []

if bond_detail_start != -1 and lewis_start != -1:
    # Determine the end of the section we want to parse (Lewis section only, or both)
    # For now, let's get both Lewis and non-Lewis
    if non_lewis_start != -1:
        # Get up to NHO DIRECTIONALITY or SECOND ORDER
        end_markers = ["NHO DIRECTIONALITY", "SECOND ORDER", "\n\n NATURAL BOND ORBITALS (Summary)"]
        detail_end = -1
        for marker in end_markers:
            pos = nbo_section.find(marker, non_lewis_start)
            if pos != -1:
                detail_end = pos
                break
    else:
        detail_end = -1
        end_markers = ["NHO DIRECTIONALITY", "SECOND ORDER", "\n\n NATURAL BOND ORBITALS (Summary)"]
        for marker in end_markers:
            pos = nbo_section.find(marker, lewis_start)
            if pos != -1:
                detail_end = pos
                break
    
    if detail_end != -1:
        bond_detail_section = nbo_section[lewis_start:detail_end]
        
        # Split into individual bond entries
        # Each bond starts with a pattern like "   1. (2.00000) BD ( 1) H  1- H  2"
        bond_entries = re.finditer(r'\n\s+(\d+)\.\s+\(([\d.]+)\)\s+(BD\*?|LP|CR|RY)\s*\(\s*(\d+)\)\s+(.*?)(?=\n\s+\d+\.\s+\(|\Z)', bond_detail_section, re.DOTALL)
        
        for bond_entry in bond_entries:
            bond_num = int(bond_entry.group(1))
            occupancy = float(bond_entry.group(2))
            bond_type = bond_entry.group(3)
            bond_index = int(bond_entry.group(4))
            atoms_and_contrib = bond_entry.group(5)
            
            # Extract atom information from first line
            atoms_line_match = re.match(r'(.*?)(?=\n|\()', atoms_and_contrib)
            atoms_line = atoms_line_match.group(1).strip() if atoms_line_match else ""
            
            # Parse atom numbers from atoms_line (e.g., "H  1- H  2" -> [1, 2])
            atom_numbers = []
            atom_num_matches = re.findall(r'([A-Z][a-z]?)\s+(\d+)', atoms_line)
            for atom_match in atom_num_matches:
                atom_numbers.append(int(atom_match[1]))
            
            bond_info = {
                'bond_num': bond_num,
                'occupancy': occupancy,
                'bond_type': bond_type,
                'bond_index': bond_index,
                'atoms': atom_numbers,
                'contributions': []
            }
            
            # Look for atomic contributions
            # Pattern: "               ( 50.00%)   0.7071* H  1 s( 99.91%)p 0.00(  0.09%)"
            contrib_pattern = r'\(\s*([\d.]+)%\)\s+([-\d.]+)\*\s+([A-Z][a-z]?)\s+(\d+)\s+s\(\s*([\d.]+)%\)p\s*([\d.]+)\(\s*([\d.]+)%\)'
            contributions = re.finditer(contrib_pattern, atoms_and_contrib)
            
            for contrib in contributions:
                bond_info['contributions'].append({
                    'percentage': float(contrib.group(1)),
                    'coefficient': float(contrib.group(2)),
                    'atom': contrib.group(3),
                    'atom_num': int(contrib.group(4)),
                    's_percent': float(contrib.group(5)),
                    'p_power': float(contrib.group(6)),
                    'p_percent': float(contrib.group(7))
                })
            
            bond_orbital_details.append(bond_info)
        
        print(f"Extracted {len(bond_orbital_details)} detailed bond orbital properties")
        if bond_orbital_details:
            for i, bond in enumerate(bond_orbital_details[:3]):
                print(f"\nBond {i+1}: {bond['bond_type']}({bond['bond_index']}) - atoms: {bond['atoms']}")
                print(f"  Occupancy: {bond['occupancy']}")
                if bond['contributions']:
                    for contrib in bond['contributions']:
                        print(f"  - {contrib['atom']}{contrib['atom_num']}: {contrib['percentage']}% (coeff: {contrib['coefficient']:.4f})")
                        print(f"    Hybridization: s({contrib['s_percent']:.2f}%)p^{contrib['p_power']:.2f}({contrib['p_percent']:.2f}%)")
            if len(bond_orbital_details) > 3:
                print(f"\n... and {len(bond_orbital_details) - 3} more bonds")
    else:
        print("Could not determine end of bond detail section")
        bond_orbital_details = []
else:
    print("Could not find bond orbital detail section")
    bond_orbital_details = []

Extracted 12 detailed bond orbital properties

Bond 1: BD(1) - atoms: [1, 2]
  Occupancy: 2.0
  - H1: 50.0% (coeff: 0.7071)
    Hybridization: s(99.91%)p^0.00(0.09%)
  - H2: 50.0% (coeff: 0.7071)
    Hybridization: s(99.91%)p^0.00(0.09%)

Bond 2: BD*(1) - atoms: [1, 2]
  Occupancy: 0.0
  - H1: 50.0% (coeff: 0.7071)
    Hybridization: s(99.91%)p^0.00(0.09%)
  - H2: 50.0% (coeff: -0.7071)
    Hybridization: s(99.91%)p^0.00(0.09%)

Bond 3: RY(1) - atoms: [1]
  Occupancy: 0.0

... and 9 more bonds


In [36]:
# Extract NHO Directionality and Bond Bending
# Find the directionality subsection
dir_start = nbo_section.find("NHO DIRECTIONALITY AND BOND BENDING")
dir_end = nbo_section.find("\n SECOND ORDER PERTURBATION", dir_start)
if dir_end == -1:
    dir_end = nbo_section.find("\n\n NATURAL BOND ORBITALS", dir_start)

nbo_directionality = []

if dir_start != -1 and dir_end != -1:
    dir_section = nbo_section[dir_start:dir_end]
    
    # Check if there are entries exceeding thresholds
    if "None exceeding thresholds" in dir_section:
        print("NHO Directionality: No entries exceeding thresholds (all bonds are linear)")
    else:
        # Parse directionality entries if they exist
        # Match atoms as: one or more occurrences of (AtomSymbol  Number) with optional dash/space separator
        # "  32. BD ( 1) N  4- C  5   78.9   53.4    79.7   50.0   3.5   102.2  234.7   1.6"
        dir_lines = re.findall(r'\n\s+(\d+)\.\s+(BD|LP|CR)\s*\(\s*(\d+)\)\s+([A-Z][a-z]?\s+\d+(?:\s*-\s*[A-Z][a-z]?\s+\d+)*)\s+([\d.-]+)\s+([\d.-]+)\s+([\d.-]+)\s+([\d.-]+)\s+([\d.-]+)', dir_section)
        
        for line_data in dir_lines:
            atoms_string = line_data[3].strip()
            
            # Parse atom numbers from atoms_string (e.g., "N  4- C  5" -> [4, 5] or "Ni  2" -> [2])
            atom_numbers = []
            atom_num_matches = re.findall(r'([A-Z][a-z]?)\s+(\d+)', atoms_string)
            for atom_match in atom_num_matches:
                atom_numbers.append(int(atom_match[1]))
            
            nbo_directionality.append({
                'nbo_num': int(line_data[0]),
                'nbo_type': line_data[1],
                'bond_index': int(line_data[2]),
                'atoms': atom_numbers,  # Now a list of atom numbers
                'line_theta': float(line_data[4]) if line_data[4] != '--' else None,
                'line_phi': float(line_data[5]) if line_data[5] != '--' else None,
                'hybrid_theta': float(line_data[6]) if line_data[6] != '--' else None,
                'hybrid_phi': float(line_data[7]) if line_data[7] != '--' else None,
                'deviation': float(line_data[8]) if line_data[8] != '--' else None
            })
        
        print(f"Extracted {len(nbo_directionality)} bond directionality entries")
        if nbo_directionality:
            for entry in nbo_directionality[:3]:
                dev_str = f"{entry['deviation']:.2f}°" if entry['deviation'] is not None else "N/A"
                print(f"  {entry['nbo_type']}({entry['bond_index']}) atoms {entry['atoms']}: deviation = {dev_str}")
else:
    print("Could not find NHO Directionality section")
    nbo_directionality = []

NHO Directionality: No entries exceeding thresholds (all bonds are linear)


In [37]:
# Extract Second-Order Perturbation Theory Analysis
# Find the second-order perturbation subsection
second_start = nbo_section.find("SECOND ORDER PERTURBATION THEORY ANALYSIS")
second_end = nbo_section.find("\n\n NATURAL BOND ORBITALS (Summary):", second_start)
if second_end == -1:
    second_end = nbo_section.find("\n $CHOOSE", second_start)

second_order_perturbation = []

if second_start != -1 and second_end != -1:
    second_section = nbo_section[second_start:second_end]
    
    # Check if there are any interactions above threshold
    if "None above threshold" in second_section or "within unit" not in second_section:
        print("Second-Order Perturbation: No interactions above threshold")
    else:
        # Parse donor-acceptor interactions
        # Pattern: "   1. BD ( 1) H  1- H  2   ->  BD*( 1) H  3- H  4      12.34     0.56     0.078"
        # Format: donor -> acceptor E(2) E(j)-E(i) F(i,j)
        interaction_pattern = r'\n\s+(BD\*?|LP|CR|RY)\s*\(\s*(\d+)\)\s+(.*?)\s+->\s+(BD\*?|LP\*?|CR\*?|RY)\s*\(\s*(\d+)\)\s+(.*?)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)'
        interactions = re.finditer(interaction_pattern, second_section)
        
        for inter in interactions:
            second_order_perturbation.append({
                'donor_type': inter.group(1),
                'donor_index': int(inter.group(2)),
                'donor_atoms': inter.group(3).strip(),
                'acceptor_type': inter.group(4),
                'acceptor_index': int(inter.group(5)),
                'acceptor_atoms': inter.group(6).strip(),
                'e2_energy': float(inter.group(7)),  # E(2) stabilization energy in kcal/mol
                'energy_diff': float(inter.group(8)),  # E(j) - E(i) in a.u.
                'fock_element': float(inter.group(9))  # F(i,j) in a.u.
            })
        
        print(f"Extracted {len(second_order_perturbation)} second-order perturbation interactions")
        if second_order_perturbation:
            for i, interaction in enumerate(second_order_perturbation[:3]):
                print(f"  {i+1}. {interaction['donor_type']}({interaction['donor_index']}) {interaction['donor_atoms']} -> {interaction['acceptor_type']}({interaction['acceptor_index']}) {interaction['acceptor_atoms']}")
                print(f"      E(2) = {interaction['e2_energy']:.2f} kcal/mol, F(i,j) = {interaction['fock_element']:.3f} a.u.")
            if len(second_order_perturbation) > 3:
                print(f"  ... and {len(second_order_perturbation) - 3} more")
else:
    print("Could not find Second-Order Perturbation section")
    second_order_perturbation = []

Second-Order Perturbation: No interactions above threshold


In [38]:
# Extract NBO Analysis Cycle Information
# Find the analysis cycle subsection
cycle_start = nbo_section.find("NATURAL BOND ORBITAL ANALYSIS:")
cycle_end = nbo_section.find("Structure accepted:", cycle_start)
if cycle_end == -1:
    cycle_end = nbo_section.find("\n\n", cycle_start + 100)

nbo_analysis_info = {}

if cycle_start != -1 and cycle_end != -1:
    cycle_section = nbo_section[cycle_start:cycle_end + 200]
    
    # Parse the cycle line
    # Pattern: "    1     2     1.90     2.00000   0.00000      0   1   0   0     0      0"
    cycle_match = re.search(r'\n\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)', cycle_section)
    
    if cycle_match:
        nbo_analysis_info = {
            'cycle': int(cycle_match.group(1)),
            'center': int(cycle_match.group(2)),
            'threshold': float(cycle_match.group(3)),
            'lewis_occupancy': float(cycle_match.group(4)),
            'non_lewis_occupancy': float(cycle_match.group(5)),
            'num_core': int(cycle_match.group(6)),
            'num_bonds': int(cycle_match.group(7)),
            'num_non_bonding_cores': int(cycle_match.group(8)),
            'num_lone_pairs': int(cycle_match.group(9)),
            'low_occ_lewis': int(cycle_match.group(10)),
            'high_occ_non_lewis': int(cycle_match.group(11))
        }
        
        print(f"NBO Analysis Info:")
        print(f"  Cycle: {nbo_analysis_info['cycle']}, Threshold: {nbo_analysis_info['threshold']}")
        print(f"  Lewis occupancy: {nbo_analysis_info['lewis_occupancy']:.5f}")
        print(f"  Non-Lewis occupancy: {nbo_analysis_info['non_lewis_occupancy']:.5f}")
        print(f"  Number of bonds: {nbo_analysis_info['num_bonds']}")
        print(f"  Number of lone pairs: {nbo_analysis_info['num_lone_pairs']}")
    else:
        print("Could not parse NBO analysis cycle information")
else:
    print("Could not find NBO analysis section")
    nbo_analysis_info = {}

NBO Analysis Info:
  Cycle: 1, Threshold: 1.9
  Lewis occupancy: 2.00000
  Non-Lewis occupancy: 0.00000
  Number of bonds: 1
  Number of lone pairs: 0


In [ ]:
import json

# Compile all NBO data into a single dictionary
nbo_data = {
    'source_file': 'test.out',
    'analysis_type': 'NBO 7.0',
    'natural_population_analysis': {
        'atoms': atom_properties,
        'totals': total_properties
    },
    'natural_atomic_orbitals': nao_properties,
    'natural_bond_orbitals': nbo_properties,
    'bond_orbital_details': bond_orbital_details,
    'nho_directionality': nbo_directionality,
    'second_order_perturbation': second_order_perturbation,
    'analysis_info': nbo_analysis_info
}

# Save to JSON file
output_filename = 'nbo_properties.json'
with open(output_file_to_analyze.parent / output_filename, 'w') as json_file:
    json.dump(nbo_data, json_file, indent=2)

✓ Saved all NBO data to 'nbo_properties.json'
  File size: 8143 bytes

Data structure saved:
  - Natural Population Analysis: 2 atoms
  - Natural Atomic Orbitals: 12 orbitals
  - Natural Bond Orbitals: 12 orbitals
  - Bond Orbital Details: 12 bonds
  - NHO Directionality: 0 entries
  - Second Order Perturbation: 0 interactions
  - Analysis Info: 11 parameters
